# Resolución de Entidades: Desambiguación de Afiliaciones

## Planteamiento del Problema

Para el caso de las `afiliaciones`, no hay duplicados exactos como tal sino que varias de ellas están escritas de forma diferente, es decir, hay representaciones alternativas de una misma afiliación.

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import unicodedata
from ast import literal_eval
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

In [2]:
DATA_DIR = Path("data/entities")
OUTPUT_DIR = Path("data/extra")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

affiliations_path = DATA_DIR / "afiliaciones_ecuador.csv"
if not affiliations_path.exists():
    affiliations_path = Path("afiliaciones_ecuador.csv")

df_aff = pd.read_csv(affiliations_path)

if "affilname_es" not in df_aff.columns:
    if "affilname" not in df_aff.columns:
        raise KeyError("Expected either 'affilname_es' or 'affilname' in the affiliations file.")
    df_aff["affilname_es"] = df_aff["affilname"]

print(f"Archivo de afiliaciones: {affiliations_path}")
print(f"Forma de df_aff: {df_aff.shape}")
print(f"Total de afiliaciones unicas: {df_aff['afid'].nunique()}")
df_aff.head()

Archivo de afiliaciones: data\entities\afiliaciones_ecuador.csv
Forma de df_aff: (8143, 5)
Total de afiliaciones unicas: 8143


,afid,affilname,affiliation-city,affiliation-country,affilname_es
0,60278953,Universidad Bolivariana del Ecuador,Duran,Ecuador,Universidad Bolivariana del Ecuador
1,133242834,Instituto de Investigación Multidisciplinaria ...,NaN,Ecuador,Instituto de Investigación Multidisciplinaria ...
2,60108912,Universidad Técnica de Manabí,Portoviejo,Ecuador,Universidad Técnica de Manabí
3,131940052,Unversidad Estatal de Milagro,Milagro,Ecuador,Unversidad Estatal de Milagro
4,60072064,Universidad Técnica Particular de Loja,Loja,Ecuador,Universidad Técnica Particular de Loja


In [3]:
df_aff.isnull().sum()

afid                      0
affilname                 0
affiliation-city       2355
affiliation-country       0
affilname_es              0
dtype: int64

### Caso: Escuela Politécnica Nacional (EPN)

In [4]:
similar_epn = [
    60072054, 101306727, 115317659,
    127982025, 128310067, 131445344,
    121697195, 132291228, 131334134,
    122120542, 123346277, 123988974,
    123789075, 124101307, 130722865,
    130722929, 130838989, 130472131,
]

In [5]:
df_aff[df_aff["afid"].isin(similar_epn)]

,afid,affilname,affiliation-city,affiliation-country,affilname_es
71,60072054,Escuela Politécnica Nacional,Quito,Ecuador,Escuela Politécnica Nacional
1743,131445344,Escuela Politécnic a Nacional,Quito,Ecuador,Escuela Politécnic a Nacional
1766,115317659,Escuela Poliécnica Nacional,Quito,Ecuador,Escuela Poliécnica Nacional
2308,132291228,Escuela Politécnica Nacional (National Polytec...,NaN,Ecuador,Escuela Politécnica Nacional (National Polytec...
2340,123346277,Escuela Poltécnica Nacional,Quito,Ecuador,Escuela Poltécnica Nacional
3119,130472131,Politechnical National School,Toledo,Ecuador,Politechnical National School
3190,121697195,Escuela Politécnica,NaN,Ecuador,Escuela Politécnica
3547,128310067,Escuela PolitCrossed D Sign©cnica Nacional,Quito,Ecuador,Escuela PolitCrossed D Sign©cnica Nacional
3590,127982025,Escuela Politcnica,NaN,Ecuador,Escuela Politcnica
3993,131334134,Escuela Politéctnica Nacional,Quito,Ecuador,Escuela Politéctnica Nacional


In [6]:
df_aff[df_aff["afid"].isin(similar_epn)].to_csv(
    OUTPUT_DIR / "afiliaciones_similares_epn.csv",
    index=False,
)

### Caso: Universidad San Francisco de Quito (USFQ)

In [7]:
similar_usfq = [60072059, 128621323, 114370973, 132049211, 106617475, 132731084, 133252427, 133107961, 105478164]

df_aff[df_aff["afid"].isin(similar_usfq)]

,afid,affilname,affiliation-city,affiliation-country,affilname_es
13,60072059,Universidad San Francisco de Quito,Quito,Ecuador,Universidad San Francisco de Quito
405,133107961,Jorge Álvarez SIME Sistemas Médicos USFQ,Quito,Ecuador,Jorge Álvarez SIME Sistemas Médicos USFQ
471,114370973,Universidad San Francisco de Quinto,Quito,Ecuador,Universidad San Francisco de Quinto
869,132731084,USFQ Datahub,Cumbayá,Ecuador,USFQ Datahub
945,128621323,USFQ,NaN,Ecuador,USFQ
1357,133252427,Laboratorio de Biología Evolutiva - USFQ,Quito,Ecuador,Laboratorio de Biología Evolutiva - USFQ
1464,132049211,San Francisco de Quito University,Cumbayá,Ecuador,San Francisco de Quito University
1607,106617475,University of San Francisco de Quito,Quito,Ecuador,University of San Francisco de Quito
7882,105478164,University of San Fransisco,NaN,Ecuador,University of San Fransisco


### Caso: Pontificia Universidad Católica del Ecuador (PUCE)

In [8]:
similar_puce = [
    60072063,
    131553112,
    114833891,
    133156531,
    123368997,
    122712001,
    127632151,
    120708320,
    124925250,
    124057755,
    121379562,
    129408339,
    122229431,
    118107657,
    120278674,
    120180587,
    117155384,
    108558436,
    112908677,
    107482755,
    108267144,
    126587005,
    101008996,
    131793873,
    132193754,
    100862184,
    100741270,
    122559898
]

df_aff[df_aff["afid"].isin(similar_puce)]

,afid,affilname,affiliation-city,affiliation-country,affilname_es
47,60072063,Pontificia Universidad Católica del Ecuador,Quito,Ecuador,Pontificia Universidad Católica del Ecuador
633,100862184,Universidad Católica,Quito,Ecuador,Universidad Católica
736,131553112,Potinficia Universidad Católica del Ecuador,NaN,Ecuador,Potinficia Universidad Católica del Ecuador
789,114833891,Pontifical University of Ecuador,Quito,Ecuador,Pontifical University of Ecuador
1306,133156531,Catholic University of Ecuador (PUCE),NaN,Ecuador,Catholic University of Ecuador (PUCE)
2331,132193754,International Relations from Pontificia Univer...,NaN,Ecuador,International Relations from Pontificia Univer...
2486,131793873,Universidad Católica del Ecuador,Riobamba,Ecuador,Universidad Católica del Ecuador
2664,112908677,Universidad Católica de Quito,Quito,Ecuador,Universidad Católica de Quito
3378,123368997,Universidad Católica del Ecuador (PUCE),NaN,Ecuador,Universidad Católica del Ecuador (PUCE)
3854,122712001,Pontifical University Catholic of Ecuador,Quito,Ecuador,Pontifical University Catholic of Ecuador


### Caso: Universidad de las Fuerzas Armadas (ESPE)

In [9]:
similar_espe = [
    133323659,
    133151254,
    122379786,
    116501774,
    130739870,
    124094683,
    122019323,
    112495846,
    109665902,
    101582946,
    60104598,
    132413089,
    128956608,
    101219483,
    131453317,
    126169807,
    131925008,
    115392390,
    117522415,
    128529271,
    118330150,
    122733307,
    129475995,
    127337623,
    125379323,
    125776733,
    116155523,
    122490118,
    119050708,
    116600643,
]

df_aff[df_aff["afid"].isin(similar_espe)]

,afid,affilname,affiliation-city,affiliation-country,affilname_es
42,60104598,Universidad de las Fuerzas Armadas ESPE,Sangolquí,Ecuador,Universidad de las Fuerzas Armadas ESPE
44,133323659,Escuela Superior Politécnica del Ejercito,Latacunga,Ecuador,Escuela Superior Politécnica del Ejercito
56,133151254,Polytechnic School of the Army ESPE,Sangolquí,Ecuador,Polytechnic School of the Army ESPE
1055,132413089,University of the Armed Forced ESPE,Salgolquí,Ecuador,University of the Armed Forced ESPE
1106,128956608,Universidad de las Fuerzas Armadas del Ecuador...,Sangolquí,Ecuador,Universidad de las Fuerzas Armadas del Ecuador...
1364,122379786,Escuela Superior Politécnica del Ejército,Sangolqui,Ecuador,Escuela Superior Politécnica del Ejército
1638,101219483,University of the Armed Forces,Quito,Ecuador,University of the Armed Forces
1708,131453317,ESPE University,NaN,Ecuador,ESPE University
1893,126169807,ESPE,Sangolquí,Ecuador,ESPE
1922,116501774,Universidad Politécnica del Ejército,Quito,Ecuador,Universidad Politécnica del Ejército


### Caso: Escuela Superior Politecnica del Litoral Ecuador (ESPOL)

In [10]:
similar_espol = [
    60072061,
    132375923,
    126135406,
    128883821,
    128747316,
    127808086,
    131258983,
    128541175,
    121418361,
    129771144,
    127303794,
    126714769,
    125790929,
    125790614,
    101913276,
    114912486,
    113527910,
    109474999,
    117876864,
    121697195
]

df_aff[df_aff["afid"].isin(similar_espol)]

,afid,affilname,affiliation-city,affiliation-country,affilname_es
12,60072061,Escuela Superior Politecnica del Litoral Ecuador,Guayaquil,Ecuador,Escuela Superior Politecnica del Litoral Ecuador
1333,113527910,Escuela Superior Politécnica del Ecuador,Guayaquil,Ecuador,Escuela Superior Politécnica del Ecuador
1563,117876864,Escuela Superior Politécnica,NaN,Ecuador,Escuela Superior Politécnica
2571,132375923,Universidad Politécnica del Litoral,NaN,Ecuador,Universidad Politécnica del Litoral
2911,126135406,Escuela Superior Politécnicadel Litoral,NaN,Ecuador,Escuela Superior Politécnicadel Litoral
3190,121697195,Escuela Politécnica,NaN,Ecuador,Escuela Politécnica
3469,128883821,Escuela Superior PolitÃľcnica del Litoral,Guayaquil,Ecuador,Escuela Superior PolitÃľcnica del Litoral
4048,128747316,Escuela Politécnica del Litoral,NaN,Ecuador,Escuela Politécnica del Litoral
4639,127808086,Escuela Superior Polit cnica Del Litoral ESPOL,NaN,Ecuador,Escuela Superior Polit cnica Del Litoral ESPOL
4741,131258983,Escuela Superior Polit cnica Del Litoral,Guayaquil,Ecuador,Escuela Superior Polit cnica Del Litoral


Otros ejemplos también son:

In [11]:
df_aff[
    df_aff["afid"].isin(
        [132217264, 132462163, 126803765, 133300960, 100358737, 132157804]
    )
]

,afid,affilname,affiliation-city,affiliation-country,affilname_es
988,132462163,Academia de Guerra del Ejército,Sangolquí,Ecuador,Academia de Guerra del Ejército
2335,132217264,Academia de Guerra del Ejercito,Quito,Ecuador,Academia de Guerra del Ejercito
4311,126803765,Academia de Defensa Militar,Conjunta,Ecuador,Academia de Defensa Militar
5126,133300960,Academia de Defensa Militar Conjunta,Sangolquí,Ecuador,Academia de Defensa Militar Conjunta
7798,100358737,ACUATECNOS,Guayaquil,Ecuador,ACUATECNOS
7933,132157804,Acuatecnos,Guayaguil,Ecuador,Acuatecnos


Esto puede deberse a la forma en que Scopus maneja las relaciones de las afiliaciones, ya que el nombre y la ciudad puede no variar, sin embargo, si puede tratarse de una sucursal de esa afiliación, por lo cual se puede definir los siguientes errores u "excepciones":
- **Variaciones de idioma**
- **Errores tipográficos**
- **Uso de acrónimos**
- **Inconsistencia en ubicación geográfica**
- **Jerarquia de instituciones**, es decir, se pueden encontrar unidaes/facultades que pertenecen a una institución padre

## Exploración de Datos

### Longitud de nombres

In [12]:
df_aff["name_length"] = df_aff["affilname_es"].str.len()
print(df_aff["name_length"].describe())
df_aff = df_aff.drop(columns=["name_length"])

count    8143.000000
mean       35.811249
std        20.452325
min         2.000000
25%        22.000000
50%        32.000000
75%        46.000000
max       218.000000
Name: name_length, dtype: float64


### Caracteres especiales

In [13]:
special_chars = df_aff[df_aff["affilname_es"].str.contains(r"[^A-Za-zÁÉÍÓÚáéíóúÑñÜü0-9 ]", na=False)]
print(f"Afiliaciones con caracteres especiales (sin contar acentos): {len(special_chars)}")
special_chars

Afiliaciones con caracteres especiales (sin contar acentos): 2266


,afid,affilname,affiliation-city,affiliation-country,affilname_es
21,133088229,Centro de Investigación y Desarrollo en Nanote...,Guayaquil,Ecuador,Centro de Investigación y Desarrollo en Nanote...
37,60104441,Universidad de las Americas - Ecuador,Quito,Ecuador,Universidad de las Americas - Ecuador
41,60113878,"Universidad Nacional de Educación, Ecuador",Azogues,Ecuador,"Universidad Nacional de Educación, Ecuador"
45,133323533,Ministerio de Ecucación del Ecuador (MINEDUC),Ambato,Ecuador,Ministerio de Ecucación del Ecuador (MINEDUC)
50,128862238,Universidad Autónoma de los Andes (UNIANDES),Quevedo,Ecuador,Universidad Autónoma de los Andes (UNIANDES)
...,...,...,...,...,...
8122,117679382,Faculté D'agronomie Et De Médicine Vétérinaire...,Quito,Ecuador,Faculté D'agronomie Et De Médicine Vétérinaire...
8125,117678904,Université Centrale de L'equateur,Quito,Ecuador,Université Centrale de L'equateur
8128,101357107,Instituto Nacional de Higiene 'Leopoldo Izquie...,NaN,Ecuador,Instituto Nacional de Higiene 'Leopoldo Izquie...
8129,114594823,Anglo-Ecuadorian Oilfields Ltd,NaN,Ecuador,Anglo-Ecuadorian Oilfields Ltd


In [14]:
with_numbers = df_aff[df_aff["affilname_es"].str.contains(r"[0-9]", na=False)]
with pd.option_context("display.max_rows", None):
    print(f"Afiliaciones con números: {len(with_numbers)}")
    with_numbers = with_numbers.sort_values("affilname_es")
    display(with_numbers)

Afiliaciones con números: 181


,afid,affilname,affiliation-city,affiliation-country,affilname_es
5834,126155917,13D03 Jipijapa-Puerto López. Ministerio de Sal...,Jipijapa,Ecuador,13D03 Jipijapa-Puerto López. Ministerio de Sal...
7726,115652939,16-310 Quito,NaN,Ecuador,16-310 Quito
5490,122819394,360Life Technologies,Quito,Ecuador,360Life Technologies
5050,124376057,3A Composites Research and Development,Guayaquil,Ecuador,3A Composites Research and Development
107,129115108,3Diversity,"Quito, Pichincha",Ecuador,3Diversity
1042,129674606,3Diversity,Quito,Ecuador,3Diversity
4129,128914072,7CargoCorp,Guayaquil,Ecuador,7CargoCorp
6623,122910175,AInstituto Tecnológico Superior 17 de Julio-Ya...,NaN,Ecuador,AInstituto Tecnológico Superior 17 de Julio-Ya...
4305,126587795,Agregado I. Categoría SENESCYT REG-INV-17-02036,Machala,Ecuador,Agregado I. Categoría SENESCYT REG-INV-17-02036
3876,127025813,AgroG2Ec S.A.,Quito,Ecuador,AgroG2Ec S.A.


La mayoria de afiliaciones que cuentan con numeros en el campo `affilname` en realidad son direcciones fisicas.  
Posibles Affs (con números) validos:  
106760808, 132621188, 114129821, 119933429, 114347231, 119084753, 131585760, 114783230, 114334903, 115847150, 116363379, 127241392, 122910175, 118184979, 123222881, 130054373, 121019933, 117876872, 120560792, 120561484, 120871990, 122161570, 126155917, 124335735, 121544468, 122969618, 123095409, 123342062, 131296976, 122819394, 124520204, 123399984, 130768137, 123658550, 124817166, 124142001, 124376057, 125063323, 124998779, 125108503, 127163551, 127003733, 127283381, 112568774, 127393723, 125686069, 125790044, 126364778, 125643866, 126413410, 126587795, 126245280, 126889915, 126291546, 127357760, 127579876, 127579955, 125781106, 100942116, 128914072, 122928960, 131167099, 129064944, 129217139, 127814983, 127025813, 128190086, 128186625, 126950111,  100729671, 128075467, 127013362, 128615871, 128828308, 129217411, 129488444, 121022423, 129856019, 129821360, 129774506, 126774636, 131200354, 124151922, 130001440, 130383661, 130496643, 116600646, 132228596, 131998230, 132785370, 132608539, 119056721, 131352547, 127756530, 130950602, 131096996, 126508162, 132235420, 132291696, 126571443, 129674606, 132478571, 132441873, 132708510, 132348260, 132348260, 132313451, 133286310, 133239592, 132684858, 133107458, 125108474, 129115108, 132830147, 133293118

### Palabras más frecuentes

In [15]:
all_words = " ".join(df_aff["affilname_es"]).lower().split()
common_words = Counter(all_words).most_common(100)
print(common_words)

[('de', 3469), ('hospital', 814), ('y', 704), ('del', 682), ('of', 635), ('instituto', 613), ('universidad', 507), ('la', 443), ('ecuador', 440), ('centro', 427), ('nacional', 354), ('salud', 342), ('superior', 338), ('and', 333), ('en', 298), ('university', 260), ('investigación', 255), ('general', 227), ('research', 223), ('tecnológico', 221), ('unidad', 209), ('fundación', 207), ('the', 202), ('ministerio', 190), ('médico', 167), ('for', 165), ('institute', 159), ('escuela', 153), ('educativa', 153), ('el', 144), ('para', 141), ('national', 140), ('center', 138), ('pública', 130), ('san', 121), ('ecuatoriana', 120), ('clínica', 116), ('ciencias', 115), ('grupo', 114), ('manabí', 111), ('departamento', 111), ('quito', 108), ('guayaquil', 108), ('facultad', 107), ('investigaciones', 102), ('social', 102), ('s.a.', 102), ('health', 99), ('desarrollo', 98), ('ecuatoriano', 89), ('ecuadorian', 88), ('foundation', 87), ('especialidades', 84), ('e', 79), ('iess', 79), ('politécnica', 78), 

Posibles categorías identificadas: hospital, instituto, universidad, centro, fundación, ministerio, escuela, clinica, grupo, departamento, facultad, laboratorio, corporación, sociedad, otros.

In [16]:
print(len(all_words))
print(len(set(all_words)))

38662
7648


## Normalizar

In [17]:
def normalize_text(text):
    text = text.lower()
    # Quitar acentos y simbolos raros
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("utf-8")
    # Quitar numeros
    text = re.sub(r"[^a-z\s]", " ", text)
    # Colapsar espacios multiples
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [18]:
df_aff["affil_clean"] = df_aff["affilname_es"].apply(normalize_text)
df_aff[df_aff["afid"].isin(similar_epn)].head()

,afid,affilname,affiliation-city,affiliation-country,affilname_es,affil_clean
71,60072054,Escuela Politécnica Nacional,Quito,Ecuador,Escuela Politécnica Nacional,escuela politecnica nacional
1743,131445344,Escuela Politécnic a Nacional,Quito,Ecuador,Escuela Politécnic a Nacional,escuela politecnic a nacional
1766,115317659,Escuela Poliécnica Nacional,Quito,Ecuador,Escuela Poliécnica Nacional,escuela poliecnica nacional
2308,132291228,Escuela Politécnica Nacional (National Polytec...,NaN,Ecuador,Escuela Politécnica Nacional (National Polytec...,escuela politecnica nacional national polytech...
2340,123346277,Escuela Poltécnica Nacional,Quito,Ecuador,Escuela Poltécnica Nacional,escuela poltecnica nacional


In [19]:
acronimos = (
    df_aff["affilname_es"]
    .str.findall(r"\(([A-Z](?:\.?[A-Z]){1,})\)")
    .explode()
    .dropna()
    .drop_duplicates()
)
acronimos = acronimos.sort_values().reset_index(drop=True)
print(f"Número de acrónimos únicos encontrados: {len(acronimos)}")
acronimos

Número de acrónimos únicos encontrados: 340


0          ABG
1        ABREC
2          ACE
3       AEPPBE
4      AEPROVI
        ...   
335       UTEG
336     UTLVTE
337        UTN
338         WP
339        WWF
Name: affilname_es, Length: 340, dtype: object

In [20]:
with pd.option_context("display.max_rows", None):
    display(df_aff["affiliation-city"].dropna().value_counts())

affiliation-city
Quito                                                       2175
Guayaquil                                                    870
Cuenca                                                       363
Ambato                                                       180
Riobamba                                                     148
Loja                                                         126
Ibarra                                                        65
Santo Domingo                                                 63
Portoviejo                                                    60
Manta                                                         57
Quevedo                                                       51
Esmeraldas                                                    50
Machala                                                       45
Latacunga                                                     41
Santa Elena                                                   31
Manabí  

In [21]:
STOPWORDS = {"de","del","la","el","los","las","y","e","en","para","por","a","al"}

def tokens(text: str):
    toks = [t for t in text.split() if t not in STOPWORDS]
    return " ".join(toks)

## FingerPrint

In [22]:
def fingerprint(text):
    if not text:
        return ""
    
    text = normalize_text(text)
    tokens = text.split(" ")
    tokens = set(tokens)
    tokens = sorted(tokens)
    return " ".join(tokens)

In [23]:
df_aff["fingerprint"] = df_aff["affilname_es"].apply(fingerprint)
df_aff[df_aff["afid"].isin(similar_epn)].head()

,afid,affilname,affiliation-city,affiliation-country,affilname_es,affil_clean,fingerprint
71,60072054,Escuela Politécnica Nacional,Quito,Ecuador,Escuela Politécnica Nacional,escuela politecnica nacional,escuela nacional politecnica
1743,131445344,Escuela Politécnic a Nacional,Quito,Ecuador,Escuela Politécnic a Nacional,escuela politecnic a nacional,a escuela nacional politecnic
1766,115317659,Escuela Poliécnica Nacional,Quito,Ecuador,Escuela Poliécnica Nacional,escuela poliecnica nacional,escuela nacional poliecnica
2308,132291228,Escuela Politécnica Nacional (National Polytec...,NaN,Ecuador,Escuela Politécnica Nacional (National Polytec...,escuela politecnica nacional national polytech...,epn escuela nacional national politecnica poly...
2340,123346277,Escuela Poltécnica Nacional,Quito,Ecuador,Escuela Poltécnica Nacional,escuela poltecnica nacional,escuela nacional poltecnica


In [24]:
fingerprint_groups = (
    df_aff
    .groupby("fingerprint")
    .filter(lambda x: len(x) > 1)
)

fingerprint_groups.sort_values("fingerprint").head(20)

,afid,affilname,affiliation-city,affiliation-country,affilname_es,affil_clean,fingerprint
1732,131412737,Casta Roja Agroindustrial S.A.,Guayaquil,Ecuador,Casta Roja Agroindustrial S.A.,casta roja agroindustrial s a,a agroindustrial casta roja s
5619,122833189,Casta roja agroindustrial S. A,NaN,Ecuador,Casta roja agroindustrial S. A,casta roja agroindustrial s a,a agroindustrial casta roja s
5091,123865589,AndinaGestión S.A.,Quito,Ecuador,AndinaGestión S.A.,andinagestion s a,a andinagestion s
4489,126100122,AndinaGestión S.A.,Quito,Ecuador,AndinaGestión S.A.,andinagestion s a,a andinagestion s
5093,123864947,AndinaGestión S.A,Quito,Ecuador,AndinaGestión S.A,andinagestion s a,a andinagestion s
5967,108598442,Inst. Nac. Autonomo Invest. A.,Quito,Ecuador,Inst. Nac. Autonomo Invest. A.,inst nac autonomo invest a,a autonomo inst invest nac
6936,101152082,Inst. Nac. Autonomo Invest. A.,Guayaquil,Ecuador,Inst. Nac. Autonomo Invest. A.,inst nac autonomo invest a,a autonomo inst invest nac
2733,124315256,Bira Bienes Raíces S.A. (BIRA S.A.),Zaruma,Ecuador,Bira Bienes Raíces S.A. (BIRA S.A.),bira bienes raices s a bira s a,a bienes bira raices s
5215,124095038,BIRA Bienes Raíces S.A.,NaN,Ecuador,BIRA Bienes Raíces S.A.,bira bienes raices s a,a bienes bira raices s
1548,131894774,Manejo y Conservación de Recursos Naturales S....,Guayaquil,Ecuador,Manejo y Conservación de Recursos Naturales S....,manejo y conservacion de recursos naturales s a s,a conservacion de manejo naturales recursos s y


In [25]:
fingerprint_groups[fingerprint_groups["afid"].isin(similar_epn)].head()

,afid,affilname,affiliation-city,affiliation-country,affilname_es,affil_clean,fingerprint
4829,124101307,National Polytechnic School (EPN),Ladrón the Guevara,Ecuador,National Polytechnic School (EPN),national polytechnic school epn,epn national polytechnic school
5249,130722865,National Polytechnic School (EPN),Ladrón the Guevara,Ecuador,National Polytechnic School (EPN),national polytechnic school epn,epn national polytechnic school
5252,130722929,National Polytechnic School (EPN),Ladrón de Guevara,Ecuador,National Polytechnic School (EPN),national polytechnic school epn,epn national polytechnic school


## TF-IDF

In [26]:
unique_names = df_aff["affilname_es"].dropna().unique()
unique_names

array(['Universidad Bolivariana del Ecuador',
       'Instituto de Investigación Multidisciplinaria Perspectivas Globales',
       'Universidad Técnica de Manabí', ...,
       'Quita Normal de Agricultura',
       'South American Development Company', 'Field Director'],
      shape=(7340,), dtype=object)

In [27]:
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3))
tfidf_matrix = vectorizer.fit_transform(unique_names)
cosine_sim = cosine_similarity(tfidf_matrix)

In [28]:
SIMILARITY_THRESHOLD = 0.85
rows, cols = np.where(cosine_sim > SIMILARITY_THRESHOLD)

In [29]:
G = nx.Graph()
G.add_nodes_from(unique_names)

for r, c in zip(rows, cols):
    if r != c:
        G.add_edge(unique_names[r], unique_names[c])

clusters = list(nx.connected_components(G))

In [30]:
name_counts = df_aff["affilname_es"].value_counts().to_dict()
name_map = {}

for cluster in clusters:
    cluster_list = list(cluster)

    # Logic: Pick the name with highest frequency.
    # If tie, pick the longest one (usually more complete).
    standard_name = max(cluster_list, key=lambda x: (name_counts.get(x, 0), len(x)))

    # Map every variation in the cluster to this standard name
    for name in cluster_list:
        name_map[name] = standard_name

In [31]:
df_aff["standardized_affilname"] = df_aff["affilname_es"].map(name_map)
df_aff[df_aff["afid"].isin(similar_epn)]

,afid,affilname,affiliation-city,affiliation-country,affilname_es,affil_clean,fingerprint,standardized_affilname
71,60072054,Escuela Politécnica Nacional,Quito,Ecuador,Escuela Politécnica Nacional,escuela politecnica nacional,escuela nacional politecnica,Escuela Superior Politécnica Agropecuaria de M...
1743,131445344,Escuela Politécnic a Nacional,Quito,Ecuador,Escuela Politécnic a Nacional,escuela politecnic a nacional,a escuela nacional politecnic,Escuela Superior Politécnica Agropecuaria de M...
1766,115317659,Escuela Poliécnica Nacional,Quito,Ecuador,Escuela Poliécnica Nacional,escuela poliecnica nacional,escuela nacional poliecnica,Escuela Poliécnica Nacional
2308,132291228,Escuela Politécnica Nacional (National Polytec...,NaN,Ecuador,Escuela Politécnica Nacional (National Polytec...,escuela politecnica nacional national polytech...,epn escuela nacional national politecnica poly...,Escuela Politécnica Nacional (National Polytec...
2340,123346277,Escuela Poltécnica Nacional,Quito,Ecuador,Escuela Poltécnica Nacional,escuela poltecnica nacional,escuela nacional poltecnica,Escuela Poltécnica Nacional
3119,130472131,Politechnical National School,Toledo,Ecuador,Politechnical National School,politechnical national school,national politechnical school,Politechnical National School
3190,121697195,Escuela Politécnica,NaN,Ecuador,Escuela Politécnica,escuela politecnica,escuela politecnica,Escuela Superior Politécnica Agropecuaria de M...
3547,128310067,Escuela PolitCrossed D Sign©cnica Nacional,Quito,Ecuador,Escuela PolitCrossed D Sign©cnica Nacional,escuela politcrossed d signcnica nacional,d escuela nacional politcrossed signcnica,Escuela PolitCrossed D Sign©cnica Nacional
3590,127982025,Escuela Politcnica,NaN,Ecuador,Escuela Politcnica,escuela politcnica,escuela politcnica,Escuela Politcnica
3993,131334134,Escuela Politéctnica Nacional,Quito,Ecuador,Escuela Politéctnica Nacional,escuela politectnica nacional,escuela nacional politectnica,Escuela Politéctnica Nacional


## Comparación manual antes y después de la desambiguación

Los cinco grupos definidos manualmente antes de la normalización (`similar_epn`, `similar_usfq`, `similar_puce`, `similar_espe` y `similar_espol`) se reutilizan como casos de evaluación. Primero se revisan las afiliaciones originales; luego, después de ejecutar la desambiguación por TF-IDF, se vuelven a consultar los mismos `afid` para observar cómo quedaron agrupados por `standardized_affilname`.

En esta sección, `Decision` corresponde a la salida del método de desambiguación y `Ground Truth` corresponde a la revisión manual de los ejemplos.

In [32]:
def parse_affiliation_ids(value):
    if pd.isna(value):
        return []

    try:
        parsed = literal_eval(value)
    except (ValueError, SyntaxError):
        return []

    if not isinstance(parsed, (list, tuple, set)):
        return []

    return [str(afid) for afid in parsed if pd.notna(afid)]


articles_path = DATA_DIR / "articulos_ecuador.csv"
if not articles_path.exists():
    articles_path = Path("articulos_ecuador.csv")

df_articles = pd.read_csv(articles_path)
affiliation_ids = (
    df_articles["affiliations"]
    .dropna()
    .apply(parse_affiliation_ids)
    .explode()
    .dropna()
)

affiliation_publication_counts = (
    affiliation_ids
    .value_counts()
    .rename_axis("afid")
    .reset_index(name="publication_count")
)

df_aff_counts = df_aff.copy()
df_aff_counts["afid"] = df_aff_counts["afid"].astype(str)

top_affiliations = affiliation_publication_counts.merge(
    df_aff_counts,
    on="afid",
    how="left",
)

top_affiliations.head(5)[
    ["afid", "affilname_es", "publication_count", "affiliation-city"]
]

,afid,affilname_es,publication_count,affiliation-city
0,60072061,Escuela Superior Politecnica del Litoral Ecuador,4917,Guayaquil
1,60072059,Universidad San Francisco de Quito,4783,Quito
2,60072054,Escuela Politécnica Nacional,4256,Quito
3,60072063,Pontificia Universidad Católica del Ecuador,4165,Quito
4,60104598,Universidad de las Fuerzas Armadas ESPE,3659,Sangolquí


In [33]:
top_institution_similar_affiliations = {
    "Escuela Superior Politecnica del Litoral Ecuador": similar_espol,
    "Universidad San Francisco de Quito": similar_usfq,
    "Escuela Politécnica Nacional": similar_epn,
    "Pontificia Universidad Católica del Ecuador": similar_puce,
    "Universidad de las Fuerzas Armadas ESPE": similar_espe,
}

# Mantener los ids como texto facilita el cruce con listas de Scopus leídas desde CSV.
top_institution_similar_affiliations = {
    institution: [str(afid) for afid in afids]
    for institution, afids in top_institution_similar_affiliations.items()
}

top_institution_similar_affiliations

{'Escuela Superior Politecnica del Litoral Ecuador': ['60072061',
  '132375923',
  '126135406',
  '128883821',
  '128747316',
  '127808086',
  '131258983',
  '128541175',
  '121418361',
  '129771144',
  '127303794',
  '126714769',
  '125790929',
  '125790614',
  '101913276',
  '114912486',
  '113527910',
  '109474999',
  '117876864',
  '121697195'],
 'Universidad San Francisco de Quito': ['60072059',
  '128621323',
  '114370973',
  '132049211',
  '106617475',
  '132731084',
  '133252427',
  '133107961',
  '105478164'],
 'Escuela Politécnica Nacional': ['60072054',
  '101306727',
  '115317659',
  '127982025',
  '128310067',
  '131445344',
  '121697195',
  '132291228',
  '131334134',
  '122120542',
  '123346277',
  '123988974',
  '123789075',
  '124101307',
  '130722865',
  '130722929',
  '130838989',
  '130472131'],
 'Pontificia Universidad Católica del Ecuador': ['60072063',
  '131553112',
  '114833891',
  '133156531',
  '123368997',
  '122712001',
  '127632151',
  '120708320',
  '1249

In [34]:
top_rank_lookup = (
    top_affiliations.head(5)
    .assign(publication_rank=lambda df: range(1, len(df) + 1))
    .set_index("affilname_es")[["publication_rank", "publication_count"]]
    .to_dict(orient="index")
)

manual_review_before_frames = []
manual_missing_afids = []

for institution, afids in top_institution_similar_affiliations.items():
    print(f"Antes de desambiguar: {institution} ({len(afids)} afid definidos manualmente)")

    similar_rows = df_aff_counts[df_aff_counts["afid"].isin(afids)].copy()
    found_afids = set(similar_rows["afid"])
    missing_afids = [afid for afid in afids if afid not in found_afids]

    if missing_afids:
        manual_missing_afids.extend(
            {"institution": institution, "afid": afid}
            for afid in missing_afids
        )
        print(f"  afid no encontrados en df_aff: {missing_afids}")

    similar_rows = (
        similar_rows
        .assign(
            review_stage="before_disambiguation",
            institution=institution,
            publication_rank=top_rank_lookup[institution]["publication_rank"],
            publication_count=top_rank_lookup[institution]["publication_count"],
        )
        .loc[:, [
            "review_stage",
            "publication_rank",
            "publication_count",
            "institution",
            "afid",
            "affilname_es",
            "affiliation-city",
            "affiliation-country",
        ]]
        .sort_values(["publication_rank", "affilname_es", "afid"])
    )

    display(similar_rows)
    manual_review_before_frames.append(similar_rows)

manual_review_before_disambiguation = pd.concat(
    manual_review_before_frames,
    ignore_index=True,
)
manual_missing_afids = pd.DataFrame(manual_missing_afids)

manual_review_before_disambiguation.to_csv(
    OUTPUT_DIR / "top_institution_similarity_review_before.csv",
    index=False,
)
manual_missing_afids.to_csv(
    OUTPUT_DIR / "top_institution_similarity_missing_afids.csv",
    index=False,
)

manual_review_before_disambiguation

Antes de desambiguar: Escuela Superior Politecnica del Litoral Ecuador (20 afid definidos manualmente)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,affiliation-city,affiliation-country
5245,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,125790929,Cera Escuela Superior Polit Ecnica Del Litoral...,NaN,Ecuador
5289,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,127303794,E. Superior Politécnica del Litoral de Guayaquil,Guayaquil,Ecuador
6999,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,114912486,ESPO Univ.,NaN,Ecuador
6671,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,129771144,ESPOL Guayaquil,Guayaquil,Ecuador
3190,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,121697195,Escuela Politécnica,NaN,Ecuador
4048,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128747316,Escuela Politécnica del Litoral,NaN,Ecuador
4741,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,131258983,Escuela Superior Polit cnica Del Litoral,Guayaquil,Ecuador
4639,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,127808086,Escuela Superior Polit cnica Del Litoral ESPOL,NaN,Ecuador
12,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,Guayaquil,Ecuador
3469,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128883821,Escuela Superior PolitÃľcnica del Litoral,Guayaquil,Ecuador


Antes de desambiguar: Universidad San Francisco de Quito (9 afid definidos manualmente)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,affiliation-city,affiliation-country
405,before_disambiguation,2,4783,Universidad San Francisco de Quito,133107961,Jorge Álvarez SIME Sistemas Médicos USFQ,Quito,Ecuador
1357,before_disambiguation,2,4783,Universidad San Francisco de Quito,133252427,Laboratorio de Biología Evolutiva - USFQ,Quito,Ecuador
1464,before_disambiguation,2,4783,Universidad San Francisco de Quito,132049211,San Francisco de Quito University,Cumbayá,Ecuador
945,before_disambiguation,2,4783,Universidad San Francisco de Quito,128621323,USFQ,NaN,Ecuador
869,before_disambiguation,2,4783,Universidad San Francisco de Quito,132731084,USFQ Datahub,Cumbayá,Ecuador
471,before_disambiguation,2,4783,Universidad San Francisco de Quito,114370973,Universidad San Francisco de Quinto,Quito,Ecuador
13,before_disambiguation,2,4783,Universidad San Francisco de Quito,60072059,Universidad San Francisco de Quito,Quito,Ecuador
1607,before_disambiguation,2,4783,Universidad San Francisco de Quito,106617475,University of San Francisco de Quito,Quito,Ecuador
7882,before_disambiguation,2,4783,Universidad San Francisco de Quito,105478164,University of San Fransisco,NaN,Ecuador


Antes de desambiguar: Escuela Politécnica Nacional (18 afid definidos manualmente)


...

Antes de desambiguar: Pontificia Universidad Católica del Ecuador (28 afid definidos manualmente)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,affiliation-city,affiliation-country
6874,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,100741270,Catholic University,Quito,Ecuador
1306,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,133156531,Catholic University of Ecuador (PUCE),NaN,Ecuador
2331,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,132193754,International Relations from Pontificia Univer...,NaN,Ecuador
5839,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,129408339,PONTIFICIA UNIVERSIDAD CATÓLICA DE QUITO,NaN,Ecuador
6289,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,122229431,Pintificia Universidad Católica de Ecuador,NaN,Ecuador
8003,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,101008996,Pontifica Universidad Catolica,NaN,Ecuador
5602,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,121379562,Pontifical Catholic University Del Ecuador,Santa Cruz,Ecuador
3854,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,122712001,Pontifical University Catholic of Ecuador,Quito,Ecuador
789,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,114833891,Pontifical University of Ecuador,Quito,Ecuador
5004,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,124925250,Pontificia Católica Universidad de Ecuador,NaN,Ecuador


Antes de desambiguar: Universidad de las Fuerzas Armadas ESPE (30 afid definidos manualmente)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,affiliation-city,affiliation-country
7944,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,112495846,Army Politechnical School (ESPE),NaN,Ecuador
7126,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,122019323,Army Polytechnical School (ESPE),Sangolqui,Ecuador
6811,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,116600643,Army's University,Quito,Ecuador
3953,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,122733307,ESPE,Sangolqui,Ecuador
1893,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,126169807,ESPE,Sangolquí,Ecuador
1708,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,131453317,ESPE University,NaN,Ecuador
5430,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,124094683,Escuela Politécnia del Ejercito,Quito,Ecuador
7425,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,109665902,Escuela Politécnica del Ejéricto,Sangolqui,Ecuador
7739,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,101582946,Escuela Polit́cnica del Ej́rcito,NaN,Ecuador
44,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,133323659,Escuela Superior Politécnica del Ejercito,Latacunga,Ecuador


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,affiliation-city,affiliation-country
0,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,125790929,Cera Escuela Superior Polit Ecnica Del Litoral...,NaN,Ecuador
1,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,127303794,E. Superior Politécnica del Litoral de Guayaquil,Guayaquil,Ecuador
2,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,114912486,ESPO Univ.,NaN,Ecuador
3,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,129771144,ESPOL Guayaquil,Guayaquil,Ecuador
4,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,121697195,Escuela Politécnica,NaN,Ecuador
...,...,...,...,...,...,...,...,...
100,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,101219483,University of the Armed Forces,Quito,Ecuador
101,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,115392390,University of the Armed Forces,Sangolqui,Ecuador
102,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,117522415,University of the Armed Forces,Sangolquí,Ecuador
103,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,122490118,University of the Army,Sangolqui,Ecuador


In [35]:
if "standardized_affilname" not in df_aff.columns:
    raise KeyError(
        "Ejecuta primero la sección TF-IDF para crear df_aff['standardized_affilname']."
    )

df_aff_disambiguated = df_aff.copy()
df_aff_disambiguated["afid"] = df_aff_disambiguated["afid"].astype(str)

manual_review_after_frames = []

for institution, afids in top_institution_similar_affiliations.items():
    print(f"Después de desambiguar: {institution} ({len(afids)} afid definidos manualmente)")

    similar_rows = df_aff_disambiguated[df_aff_disambiguated["afid"].isin(afids)].copy()
    anchor_afid = afids[0]
    anchor_standardized = similar_rows.loc[
        similar_rows["afid"].eq(anchor_afid),
        "standardized_affilname",
    ]
    anchor_standardized = anchor_standardized.iloc[0] if not anchor_standardized.empty else None

    similar_rows = (
        similar_rows
        .assign(
            review_stage="after_disambiguation",
            institution=institution,
            publication_rank=top_rank_lookup[institution]["publication_rank"],
            publication_count=top_rank_lookup[institution]["publication_count"],
            anchor_afid=anchor_afid,
            anchor_standardized_affilname=anchor_standardized,
            merged_with_anchor_tfidf=lambda df: df["standardized_affilname"].eq(anchor_standardized),
            algorithm_decision=lambda df: np.where(
                df["standardized_affilname"].eq(anchor_standardized),
                "Merge",
                "Separate",
            ),
        )
        .loc[:, [
            "review_stage",
            "publication_rank",
            "publication_count",
            "institution",
            "afid",
            "affilname_es",
            "standardized_affilname",
            "anchor_afid",
            "anchor_standardized_affilname",
            "merged_with_anchor_tfidf",
            "algorithm_decision",
            "affiliation-city",
            "affiliation-country",
        ]]
        .sort_values(["publication_rank", "standardized_affilname", "affilname_es", "afid"])
    )

    display(similar_rows)
    manual_review_after_frames.append(similar_rows)

manual_review_after_disambiguation = pd.concat(
    manual_review_after_frames,
    ignore_index=True,
)

manual_review_after_disambiguation.to_csv(
    OUTPUT_DIR / "top_institution_similarity_review_after.csv",
    index=False,
)

top_institution_similarity_review = manual_review_after_disambiguation.copy()
top_institution_similarity_review.to_csv(
    OUTPUT_DIR / "top_institution_similarity_review.csv",
    index=False,
)

manual_review_after_disambiguation

Después de desambiguar: Escuela Superior Politecnica del Litoral Ecuador (20 afid definidos manualmente)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,standardized_affilname,anchor_afid,anchor_standardized_affilname,merged_with_anchor_tfidf,algorithm_decision,affiliation-city,affiliation-country
5245,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,125790929,Cera Escuela Superior Polit Ecnica Del Litoral...,Cera Escuela Superior Polit Ecnica Del Litoral...,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,NaN,Ecuador
5289,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,127303794,E. Superior Politécnica del Litoral de Guayaquil,E. Superior Politécnica del Litoral de Guayaquil,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,Guayaquil,Ecuador
6999,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,114912486,ESPO Univ.,ESPO Univ.,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,NaN,Ecuador
6671,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,129771144,ESPOL Guayaquil,ESPOL Guayaquil,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,Guayaquil,Ecuador
4741,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,131258983,Escuela Superior Polit cnica Del Litoral,Escuela Superior Polit cnica Del Litoral ESPOL,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,Guayaquil,Ecuador
4639,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,127808086,Escuela Superior Polit cnica Del Litoral ESPOL,Escuela Superior Polit cnica Del Litoral ESPOL,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,NaN,Ecuador
12,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,Merge,Guayaquil,Ecuador
3469,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128883821,Escuela Superior PolitÃľcnica del Litoral,Escuela Superior PolitÃľcnica del Litoral,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,Guayaquil,Ecuador
3190,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,121697195,Escuela Politécnica,Escuela Superior Politécnica Agropecuaria de M...,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,NaN,Ecuador
4048,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128747316,Escuela Politécnica del Litoral,Escuela Superior Politécnica Agropecuaria de M...,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,NaN,Ecuador


Después de desambiguar: Universidad San Francisco de Quito (9 afid definidos manualmente)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,standardized_affilname,anchor_afid,anchor_standardized_affilname,merged_with_anchor_tfidf,algorithm_decision,affiliation-city,affiliation-country
405,after_disambiguation,2,4783,Universidad San Francisco de Quito,133107961,Jorge Álvarez SIME Sistemas Médicos USFQ,Jorge Álvarez SIME Sistemas Médicos USFQ,60072059,University of San Francisco de Quito,False,Separate,Quito,Ecuador
1357,after_disambiguation,2,4783,Universidad San Francisco de Quito,133252427,Laboratorio de Biología Evolutiva - USFQ,Laboratorio de Biología Evolutiva - USFQ,60072059,University of San Francisco de Quito,False,Separate,Quito,Ecuador
945,after_disambiguation,2,4783,Universidad San Francisco de Quito,128621323,USFQ,USFQ,60072059,University of San Francisco de Quito,False,Separate,NaN,Ecuador
869,after_disambiguation,2,4783,Universidad San Francisco de Quito,132731084,USFQ Datahub,USFQ Datahub,60072059,University of San Francisco de Quito,False,Separate,Cumbayá,Ecuador
1464,after_disambiguation,2,4783,Universidad San Francisco de Quito,132049211,San Francisco de Quito University,University of San Francisco de Quito,60072059,University of San Francisco de Quito,True,Merge,Cumbayá,Ecuador
471,after_disambiguation,2,4783,Universidad San Francisco de Quito,114370973,Universidad San Francisco de Quinto,University of San Francisco de Quito,60072059,University of San Francisco de Quito,True,Merge,Quito,Ecuador
13,after_disambiguation,2,4783,Universidad San Francisco de Quito,60072059,Universidad San Francisco de Quito,University of San Francisco de Quito,60072059,University of San Francisco de Quito,True,Merge,Quito,Ecuador
1607,after_disambiguation,2,4783,Universidad San Francisco de Quito,106617475,University of San Francisco de Quito,University of San Francisco de Quito,60072059,University of San Francisco de Quito,True,Merge,Quito,Ecuador
7882,after_disambiguation,2,4783,Universidad San Francisco de Quito,105478164,University of San Fransisco,University of San Fransisco,60072059,University of San Francisco de Quito,False,Separate,NaN,Ecuador


Después de desambiguar: Escuela Politécnica Nacional (18 afid definidos manualmente)


...

Después de desambiguar: Pontificia Universidad Católica del Ecuador (28 afid definidos manualmente)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,standardized_affilname,anchor_afid,anchor_standardized_affilname,merged_with_anchor_tfidf,algorithm_decision,affiliation-city,affiliation-country
6874,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,100741270,Catholic University,Catholic University,60072063,Pontificia Universidad Católica,False,Separate,Quito,Ecuador
1306,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,133156531,Catholic University of Ecuador (PUCE),Catholic University of Ecuador (PUCE),60072063,Pontificia Universidad Católica,False,Separate,NaN,Ecuador
2331,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,132193754,International Relations from Pontificia Univer...,International Relations from Pontificia Univer...,60072063,Pontificia Universidad Católica,False,Separate,NaN,Ecuador
8003,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,101008996,Pontifica Universidad Catolica,Pontifica Universidad Catolica,60072063,Pontificia Universidad Católica,False,Separate,NaN,Ecuador
5602,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,121379562,Pontifical Catholic University Del Ecuador,Pontifical Catholic University Del Ecuador,60072063,Pontificia Universidad Católica,False,Separate,Santa Cruz,Ecuador
3854,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,122712001,Pontifical University Catholic of Ecuador,Pontifical Catholic University Del Ecuador,60072063,Pontificia Universidad Católica,False,Separate,Quito,Ecuador
789,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,114833891,Pontifical University of Ecuador,Pontifical University of Ecuador,60072063,Pontificia Universidad Católica,False,Separate,Quito,Ecuador
6638,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,120180587,Pontificia Universid Catolica del Ecuador,Pontificia Universid Catolica del Ecuador,60072063,Pontificia Universidad Católica,False,Separate,NaN,Ecuador
5558,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,122559898,Pontificia Universidad Catolica del Peril,Pontificia Universidad Catolica del Peril,60072063,Pontificia Universidad Católica,False,Separate,NaN,Ecuador
5839,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,129408339,PONTIFICIA UNIVERSIDAD CATÓLICA DE QUITO,Pontificia Universidad Católica,60072063,Pontificia Universidad Católica,True,Merge,NaN,Ecuador


Después de desambiguar: Universidad de las Fuerzas Armadas ESPE (30 afid definidos manualmente)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,standardized_affilname,anchor_afid,anchor_standardized_affilname,merged_with_anchor_tfidf,algorithm_decision,affiliation-city,affiliation-country
7944,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,112495846,Army Politechnical School (ESPE),Army Politechnical School (ESPE),133323659,Escuela Superior Politécnica del Ejercito,False,Separate,NaN,Ecuador
7126,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,122019323,Army Polytechnical School (ESPE),Army Politechnical School (ESPE),133323659,Escuela Superior Politécnica del Ejercito,False,Separate,Sangolqui,Ecuador
6811,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,116600643,Army's University,Army's University,133323659,Escuela Superior Politécnica del Ejercito,False,Separate,Quito,Ecuador
3953,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,122733307,ESPE,ESPE,133323659,Escuela Superior Politécnica del Ejercito,False,Separate,Sangolqui,Ecuador
1893,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,126169807,ESPE,ESPE,133323659,Escuela Superior Politécnica del Ejercito,False,Separate,Sangolquí,Ecuador
1708,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,131453317,ESPE University,ESPE University,133323659,Escuela Superior Politécnica del Ejercito,False,Separate,NaN,Ecuador
7425,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,109665902,Escuela Politécnica del Ejéricto,Escuela Politécnica del Ejéricto,133323659,Escuela Superior Politécnica del Ejercito,False,Separate,Sangolqui,Ecuador
7739,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,101582946,Escuela Polit́cnica del Ej́rcito,Escuela Polit́cnica del Ej́rcito,133323659,Escuela Superior Politécnica del Ejercito,False,Separate,NaN,Ecuador
5430,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,124094683,Escuela Politécnia del Ejercito,Escuela Superior Politécnica del Ejercito,133323659,Escuela Superior Politécnica del Ejercito,True,Merge,Quito,Ecuador
44,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,133323659,Escuela Superior Politécnica del Ejercito,Escuela Superior Politécnica del Ejercito,133323659,Escuela Superior Politécnica del Ejercito,True,Merge,Latacunga,Ecuador


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,standardized_affilname,anchor_afid,anchor_standardized_affilname,merged_with_anchor_tfidf,algorithm_decision,affiliation-city,affiliation-country
0,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,125790929,Cera Escuela Superior Polit Ecnica Del Litoral...,Cera Escuela Superior Polit Ecnica Del Litoral...,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,NaN,Ecuador
1,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,127303794,E. Superior Politécnica del Litoral de Guayaquil,E. Superior Politécnica del Litoral de Guayaquil,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,Guayaquil,Ecuador
2,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,114912486,ESPO Univ.,ESPO Univ.,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,NaN,Ecuador
3,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,129771144,ESPOL Guayaquil,ESPOL Guayaquil,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,Guayaquil,Ecuador
4,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,131258983,Escuela Superior Polit cnica Del Litoral,Escuela Superior Polit cnica Del Litoral ESPOL,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Separate,Guayaquil,Ecuador
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,101219483,University of the Armed Forces,University of the Armed Forces,133323659,Escuela Superior Politécnica del Ejercito,False,Separate,Quito,Ecuador
101,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,115392390,University of the Armed Forces,University of the Armed Forces,133323659,Escuela Superior Politécnica del Ejercito,False,Separate,Sangolqui,Ecuador
102,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,117522415,University of the Armed Forces,University of the Armed Forces,133323659,Escuela Superior Politécnica del Ejercito,False,Separate,Sangolquí,Ecuador
103,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,122490118,University of the Army,University of the Army,133323659,Escuela Superior Politécnica del Ejercito,False,Separate,Sangolqui,Ecuador


In [36]:
tfidf_disambiguation_group_summary = (
    manual_review_after_disambiguation
    .groupby(["institution", "standardized_affilname"], dropna=False)
    .agg(
        grouped_afids=("afid", lambda values: ", ".join(values)),
        grouped_names=("affilname_es", lambda values: " | ".join(values)),
        n_variants=("afid", "count"),
    )
    .reset_index()
    .sort_values(["institution", "n_variants"], ascending=[True, False])
)

tfidf_disambiguation_group_summary.to_csv(
    OUTPUT_DIR / "tfidf_disambiguation_group_summary.csv",
    index=False,
)

tfidf_disambiguation_group_summary

,institution,standardized_affilname,grouped_afids,grouped_names,n_variants
9,Escuela Politécnica Nacional,Escuela Superior Politécnica Agropecuaria de M...,"131445344, 121697195, 60072054",Escuela Politécnic a Nacional | Escuela Polité...,3
11,Escuela Politécnica Nacional,National Polytechnic School (EPN),"124101307, 130722865, 130722929",National Polytechnic School (EPN) | National P...,3
0,Escuela Politécnica Nacional,Escuela Poitecnica Nacional,101306727,Escuela Poitecnica Nacional,1
1,Escuela Politécnica Nacional,Escuela PolitCrossed D Sign©cnica Nacional,128310067,Escuela PolitCrossed D Sign©cnica Nacional,1
2,Escuela Politécnica Nacional,Escuela Politcnica,127982025,Escuela Politcnica,1
...,...,...,...,...,...
60,Universidad de las Fuerzas Armadas ESPE,Universidad of Armed Force ESPE,129475995,Universidad of Armed Force ESPE,1
61,Universidad de las Fuerzas Armadas ESPE,University of Armed Forces – ESPE,128529271,University of Armed Forces – ESPE,1
62,Universidad de las Fuerzas Armadas ESPE,University of the Armed Forced ESPE,132413089,University of the Armed Forced ESPE,1
64,Universidad de las Fuerzas Armadas ESPE,University of the Army,122490118,University of the Army,1


## Representative cases for affiliation disambiguation

The following table expands the five manually curated institutional cases into a full algorithm-vs-ground-truth comparison. For each manual case, the table takes every TF-IDF group touched by the manual `afid` examples and lists all members of those algorithmic groups, including affiliations not present in the manual list.

`Decision` is computed relative to the seed affiliation of each manual case: `Merge` means the grouped affiliation shares the seed's `standardized_affilname`, while `Separate` means it was placed in a different TF-IDF group. `Ground Truth` is based on the manual examples: `Same` if the grouped `afid` is in the manual list for the case and `Different` otherwise.

In [37]:
paper_case_columns = [
    "Affiliation A",
    "Affiliation B",
    "Decision",
    "Ground Truth",
    "Explanation",
]

comparison_columns = [
    "Manual Case",
    "Seed AFID",
    "Seed Affiliation",
    "Algorithm Group",
    "Algorithm Group Size",
    "Grouped AFID",
    "Grouped Affiliation",
    "In Manual Ground Truth",
] + paper_case_columns

affiliation_lookup = (
    df_aff_disambiguated
    .drop_duplicates(subset="afid")
    .set_index("afid")
)


def affiliation_value(afid, column):
    afid = str(afid)
    if afid not in affiliation_lookup.index:
        raise KeyError(f"Affiliation id {afid} was not found in the affiliations table.")
    return affiliation_lookup.at[afid, column]


def ground_truth_for_member(grouped_afid, manual_afids):
    return "Same" if str(grouped_afid) in manual_afids else "Different"


def decision_against_seed(group_standardized_affilname, seed_standardized_affilname):
    return "Merge" if group_standardized_affilname == seed_standardized_affilname else "Separate"


comparison_rows = []
algorithm_group_summary_rows = []

for manual_case, manual_afids in top_institution_similar_affiliations.items():
    manual_afids = [str(afid) for afid in manual_afids]
    manual_afid_set = set(manual_afids)
    seed_afid = manual_afids[0]
    seed_affiliation = affiliation_value(seed_afid, "affilname_es")
    seed_standardized_affilname = affiliation_value(seed_afid, "standardized_affilname")

    touched_algorithm_groups = (
        manual_review_after_disambiguation
        .loc[
            manual_review_after_disambiguation["institution"].eq(manual_case),
            "standardized_affilname",
        ]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    for algorithm_group in touched_algorithm_groups:
        algorithm_group_members = (
            df_aff_disambiguated[df_aff_disambiguated["standardized_affilname"].eq(algorithm_group)]
            .copy()
            .sort_values(["affilname_es", "afid"])
        )
        group_afids = set(algorithm_group_members["afid"].astype(str))
        manual_hits = group_afids.intersection(manual_afid_set)
        false_positive_count = len(group_afids.difference(manual_afid_set))

        algorithm_group_summary_rows.append({
            "Manual Case": manual_case,
            "Algorithm Group": algorithm_group,
            "Algorithm Group Size": len(algorithm_group_members),
            "Manual AFIDs in Group": len(manual_hits),
            "Manual AFIDs Total": len(manual_afid_set),
            "Potential False Positives": false_positive_count,
            "Seed Group": algorithm_group == seed_standardized_affilname,
        })

        for _, member in algorithm_group_members.iterrows():
            grouped_afid = str(member["afid"])
            grouped_affiliation = member["affilname_es"]
            decision = decision_against_seed(
                algorithm_group,
                seed_standardized_affilname,
            )
            ground_truth = ground_truth_for_member(grouped_afid, manual_afid_set)

            if decision == "Merge" and ground_truth == "Same":
                explanation = "True positive: algorithm grouped a manual match with the seed affiliation"
            elif decision == "Merge" and ground_truth == "Different":
                explanation = "Potential false positive: algorithm grouped an affiliation outside the manual set"
            elif decision == "Separate" and ground_truth == "Same":
                explanation = "Potential false negative: manual match placed in a different algorithm group"
            else:
                explanation = "True negative relative to the seed affiliation"

            comparison_rows.append({
                "Manual Case": manual_case,
                "Seed AFID": seed_afid,
                "Seed Affiliation": seed_affiliation,
                "Algorithm Group": algorithm_group,
                "Algorithm Group Size": len(algorithm_group_members),
                "Grouped AFID": grouped_afid,
                "Grouped Affiliation": grouped_affiliation,
                "In Manual Ground Truth": grouped_afid in manual_afid_set,
                "Affiliation A": seed_affiliation,
                "Affiliation B": grouped_affiliation,
                "Decision": decision,
                "Ground Truth": ground_truth,
                "Explanation": explanation,
            })

affiliation_disambiguation_cases = pd.DataFrame(
    comparison_rows,
    columns=comparison_columns,
)

affiliation_disambiguation_group_summary = pd.DataFrame(
    algorithm_group_summary_rows,
)

display(affiliation_disambiguation_cases)

affiliation_disambiguation_cases.to_csv(
    OUTPUT_DIR / "affiliation_disambiguation_cases.csv",
    index=False,
)
affiliation_disambiguation_group_summary.to_csv(
    OUTPUT_DIR / "affiliation_disambiguation_group_summary.csv",
    index=False,
)


def latex_escape(value):
    escape_map = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(escape_map.get(char, char) for char in str(value))


def dataframe_to_latex(df, caption, label):
    column_spec = "l" * len(df.columns)
    latex_lines = [
        r"\begin{table}",
        r"\centering",
        rf"\caption{{{latex_escape(caption)}}}",
        rf"\label{{{latex_escape(label)}}}",
        rf"\begin{{tabular}}{{{column_spec}}}",
        r"\hline",
        " & ".join(latex_escape(column) for column in df.columns) + r" \\",
        r"\hline",
    ]

    for _, row in df.iterrows():
        latex_lines.append(
            " & ".join(latex_escape(row[column]) for column in df.columns) + r" \\",
        )

    latex_lines.extend([
        r"\hline",
        r"\end{tabular}",
        r"\end{table}",
    ])
    return "\n".join(latex_lines) + "\n"


latex_table = dataframe_to_latex(
    affiliation_disambiguation_cases[paper_case_columns],
    caption="Representative cases for affiliation disambiguation.",
    label="tab:affiliation-disambiguation-cases",
)
(OUTPUT_DIR / "affiliation_disambiguation_cases.tex").write_text(
    latex_table,
    encoding="utf-8",
)

,Manual Case,Seed AFID,Seed Affiliation,Algorithm Group,Algorithm Group Size,Grouped AFID,Grouped Affiliation,In Manual Ground Truth,Affiliation A,Affiliation B,Decision,Ground Truth,Explanation
0,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,Cera Escuela Superior Polit Ecnica Del Litoral...,1,125790929,Cera Escuela Superior Polit Ecnica Del Litoral...,True,Escuela Superior Politecnica del Litoral Ecuador,Cera Escuela Superior Polit Ecnica Del Litoral...,Separate,Same,Potential false negative: manual match placed ...
1,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,E. Superior Politécnica del Litoral de Guayaquil,1,127303794,E. Superior Politécnica del Litoral de Guayaquil,True,Escuela Superior Politecnica del Litoral Ecuador,E. Superior Politécnica del Litoral de Guayaquil,Separate,Same,Potential false negative: manual match placed ...
2,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,ESPO Univ.,1,114912486,ESPO Univ.,True,Escuela Superior Politecnica del Litoral Ecuador,ESPO Univ.,Separate,Same,Potential false negative: manual match placed ...
3,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,ESPOL Guayaquil,1,129771144,ESPOL Guayaquil,True,Escuela Superior Politecnica del Litoral Ecuador,ESPOL Guayaquil,Separate,Same,Potential false negative: manual match placed ...
4,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,Escuela Superior Polit cnica Del Litoral ESPOL,2,131258983,Escuela Superior Polit cnica Del Litoral,True,Escuela Superior Politecnica del Litoral Ecuador,Escuela Superior Polit cnica Del Litoral,Separate,Same,Potential false negative: manual match placed ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,Universidad de las Fuerzas Armadas ESPE,133323659,Escuela Superior Politécnica del Ejercito,University of the Armed Forces,5,115392390,University of the Armed Forces,True,Escuela Superior Politécnica del Ejercito,University of the Armed Forces,Separate,Same,Potential false negative: manual match placed ...
147,Universidad de las Fuerzas Armadas ESPE,133323659,Escuela Superior Politécnica del Ejercito,University of the Armed Forces,5,117522415,University of the Armed Forces,True,Escuela Superior Politécnica del Ejercito,University of the Armed Forces,Separate,Same,Potential false negative: manual match placed ...
148,Universidad de las Fuerzas Armadas ESPE,133323659,Escuela Superior Politécnica del Ejercito,University of the Armed Forces,5,127781651,University of the Armed Forces,False,Escuela Superior Politécnica del Ejercito,University of the Armed Forces,Separate,Different,True negative relative to the seed affiliation
149,Universidad de las Fuerzas Armadas ESPE,133323659,Escuela Superior Politécnica del Ejercito,University of the Army,1,122490118,University of the Army,True,Escuela Superior Politécnica del Ejercito,University of the Army,Separate,Same,Potential false negative: manual match placed ...


27021